In [1]:
import altair as alt
import gcsfs
import pandas as pd


GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [4]:
GCS = "gs://calitp-analytics-data/data-analyses/ntd/"
orig_df = pd.read_parquet(
    f"{GCS}raw_transit_performance_metrics_data.parquet",
    filesystem = gcsfs.GCSFileSystem()
)
orig_df.dtypes

agency_name           object
agency_status         object
city                  object
mode                  object
service               object
ntd_id                object
reporter_type         object
reporting_module      object
state                 object
primary_uza_name      object
year                  object
upt                    int64
vrh                    int64
vrm                    int64
opexp_total            int64
RTPA                  object
_merge              category
dtype: object

In [5]:
orig_df.head(2)

,agency_name,agency_status,city,mode,service,ntd_id,reporter_type,reporting_module,state,primary_uza_name,year,upt,vrh,vrm,opexp_total,RTPA,_merge
0,City of Porterville (COLT) - Transit Department,Active,Porterville,Demand Response,Purchased Transportation,90198,Building Reporter,Urban,CA,"Porterville, CA",2019,13112,2997,43696,572799,Tulare County Association of Governments,both
1,City of Porterville (COLT) - Transit Department,Active,Porterville,Demand Response,Purchased Transportation,90198,Building Reporter,Urban,CA,"Porterville, CA",2020,11523,3669,48138,686165,Tulare County Association of Governments,both


# what columns are needed
* RTPA - shows `rtpa_name`, not `rtpa_name_split`
* just read in the subset of columns needed, these metrics are precalculated

In [15]:
crosswalk = pd.read_parquet(
    f"{GCS_FILE_PATH}crosswalk2.parquet", 
    filesystem=gcsfs.GCSFileSystem(),
    columns = ["ntd_id_2022", "rtpa_name"]
).rename(columns = {"ntd_id_2022": "ntd_id"})

df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual.parquet",
    # should only certain columns be read in? now this table is much larger
    filesystem=gcsfs.GCSFileSystem(),
    columns = [
        "source_agency", "agency_status", "source_city", 
        "mode", "type_of_service", "ntd_id", 
        "reporter_type", "reporting_module", "source_state", "primary_uza_name",
        "year", "unlinked_passenger_trips", "vehicle_revenue_hours", "vehicle_revenue_miles",
        "operating_expenses_total",
    ]
).merge(
    crosswalk,
    on = "ntd_id",
    how = "left"
)

# mode should be mode_full_name
# service refers to type_of_service_full name 
df.dtypes

source_agency               object
agency_status               object
source_city                 object
mode                        object
type_of_service             object
ntd_id                      object
reporter_type               object
reporting_module            object
source_state                object
primary_uza_name            object
year                         Int64
unlinked_passenger_trips     Int64
vehicle_revenue_hours        Int64
vehicle_revenue_miles        Int64
operating_expenses_total     Int64
rtpa_name                   object
dtype: object

In [19]:
df2 = df[df.year <= 2023].reset_index(drop=True)

In [20]:
orig_df.shape, df2.shape

((2091, 17), (2676, 16))

In [23]:
# there are some additional ones in the dbt model
m1 = pd.merge(
    orig_df[["ntd_id"]].drop_duplicates(),
    df2[["ntd_id"]].drop_duplicates(),
    on = ["ntd_id", ],
    how = "outer",
    indicator=True
)
    
m1._merge.value_counts()

_merge
both          168
right_only     15
left_only       0
Name: count, dtype: int64

In [25]:
m1[m1._merge == "right_only"]

,ntd_id,_merge
0,30109,right_only
1,30131,right_only
99,90235,right_only
101,90238,right_only
106,90249,right_only
164,90313,right_only
165,90314,right_only
170,91092,right_only
171,99256,right_only
172,99262,right_only
